# Stochastic Interest Rate Modelling and Yield Curve Reconstruction

This notebook implements a Cox-Ingersoll-Ross (CIR) based model to reconstruct the yield curve using only the observed 3M yield. The work includes a CIR baseline, a CIR++ extension, prediction generation, and out-of-sample evaluation on the provided test data.

## Data Loading and Setup

The training data contains the historical zero-coupon yield curve across multiple maturities. The test input file contains only the 3M yield, which is used as the single observable input for reconstruction. The code also cleans column names because the supplied CSV files contain leading spaces in the maturity headers.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd

TRAIN_FILE = Path("train_data.csv")
TEST_FILE = Path("test_data.csv")
TEST_3M_FILE = Path("test_data_3M.csv")

if not TRAIN_FILE.exists():
    TRAIN_FILE = Path.home() / "Downloads" / "train_data.csv"
    TEST_FILE = Path.home() / "Downloads" / "test_data.csv"
    TEST_3M_FILE = Path.home() / "Downloads" / "test_data_3M.csv"

@dataclass
class CirParams:
    kappa: float
    theta: float
    sigma: float
    dt: float

def maturity_from_col(col: str) -> float | None:
    match = re.search(r"ZC(\d+)YR", col.strip(), flags=re.I)
    return None if match is None else float(match.group(1)) / 100.0

def curve_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in df.columns if maturity_from_col(col) is not None]

def read_curve(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [col.strip() for col in df.columns]
    df["Date"] = pd.to_datetime(df["Date"])
    return df.sort_values("Date").reset_index(drop=True)

## Baseline Model: CIR Short-Rate Calibration

The baseline uses the 3M yield as a proxy for the short rate. CIR parameters are estimated using an OLS discretization of the square-root diffusion. After calibration, the analytical CIR zero-coupon bond formula converts each observed 3M rate into model-implied yields across the remaining maturities.

In [ ]:
def infer_dt(dates: pd.Series) -> float:
    gaps = dates.diff().dt.days.dropna()
    return float(gaps[gaps > 0].median()) / 365.25 if len(gaps) else 1.0 / 252.0

def fit_cir_ols(rates: pd.Series, dt: float) -> CirParams:
    r = rates.to_numpy(dtype=float).clip(1e-8)
    x = r[:-1]
    y = np.diff(r)
    design = np.column_stack([np.ones_like(x), x])
    intercept, slope = np.linalg.lstsq(design, y, rcond=None)[0]
    kappa = float(np.clip(-slope / dt, 1e-5, 30.0))
    theta = float(np.clip(intercept / (kappa * dt), 1e-6, 1.0))
    residual = y - (intercept + slope * x)
    sigma2 = np.mean((residual ** 2) / (np.maximum(x, 1e-8) * dt))
    sigma = float(np.clip(math.sqrt(max(sigma2, 1e-10)), 1e-5, 5.0))
    return CirParams(kappa, theta, sigma, dt)

def cir_zero_coupon_yields(short_rates: np.ndarray, maturities: np.ndarray, params: CirParams) -> np.ndarray:
    r = short_rates.reshape(-1, 1).clip(1e-8)
    tau = maturities.reshape(1, -1).clip(1e-8)
    gamma = math.sqrt(params.kappa ** 2 + 2 * params.sigma ** 2)
    exp_term = np.exp(np.clip(gamma * tau, -60, 60))
    denominator = (gamma + params.kappa) * (exp_term - 1) + 2 * gamma
    b_tau = 2 * (exp_term - 1) / denominator
    a_base = 2 * gamma * np.exp((params.kappa + gamma) * tau / 2) / denominator
    a_tau = np.power(np.maximum(a_base, 1e-12), 2 * params.kappa * params.theta / params.sigma ** 2)
    price = np.maximum(a_tau * np.exp(-b_tau * r), 1e-12)
    return -np.log(price) / tau

## Extension: CIR++ Smooth Residual Shift

The extension keeps the CIR baseline as the stochastic core and adds a deterministic CIR++ residual adjustment. The residual shift is fitted across calendar time and maturity using a smooth log-maturity basis. This improves curve reconstruction while still using only the same-date 3M yield as input.

In [ ]:
def time_design(short_rates: np.ndarray, start_index: int = 0, trading_days: float = 252.0) -> np.ndarray:
    t = (np.arange(len(short_rates), dtype=float) + start_index) / trading_days
    r = short_rates.astype(float)
    return np.column_stack([np.ones_like(r), r, t, r * t])

def log_maturity_basis(maturities: np.ndarray, degree: int = 3) -> np.ndarray:
    z = np.log1p(maturities.astype(float))
    return np.column_stack([z ** power for power in range(degree + 1)])

def fit_cirpp_residual_factors(
    train_short: np.ndarray,
    train_curve: np.ndarray,
    maturities: np.ndarray,
    params: CirParams,
) -> tuple[np.ndarray, np.ndarray]:
    baseline_train = cir_zero_coupon_yields(train_short, maturities, params)
    residual_curve = train_curve - baseline_train
    basis = log_maturity_basis(maturities)
    residual_factors = np.linalg.lstsq(basis, residual_curve.T, rcond=None)[0].T
    coefficients = np.linalg.lstsq(time_design(train_short), residual_factors, rcond=None)[0]
    return coefficients, basis

def predict_cirpp_residual(
    baseline_curve: np.ndarray,
    test_short: np.ndarray,
    coefficients: np.ndarray,
    basis: np.ndarray,
    start_index: int,
) -> np.ndarray:
    residual_prediction = time_design(test_short, start_index=start_index) @ coefficients @ basis.T
    return baseline_curve + residual_prediction

## Generate Predictions

This section creates the final prediction file. For every target maturity, the output contains both the baseline CIR prediction and the CIR++ extension prediction. The format is kept submission-ready with columns such as `baseline_CIR_ZC050YR` and `CIRpp_ZC050YR`.

In [ ]:
train = read_curve(TRAIN_FILE)
test = read_curve(TEST_FILE)
test_3m = read_curve(TEST_3M_FILE)

target_cols = [col for col in curve_columns(train) if col != "ZC025YR"]
maturities = np.array([maturity_from_col(col) for col in target_cols], dtype=float)

params = fit_cir_ols(train["ZC025YR"], infer_dt(train["Date"]))
baseline = cir_zero_coupon_yields(test_3m["ZC025YR"].to_numpy(dtype=float), maturities, params)
coefs, basis = fit_cirpp_residual_factors(
    train["ZC025YR"].to_numpy(dtype=float),
    train[target_cols].to_numpy(dtype=float),
    maturities,
    params,
)
cirpp = predict_cirpp_residual(baseline, test_3m["ZC025YR"].to_numpy(dtype=float), coefs, basis, start_index=len(train))

predictions = pd.DataFrame({"Date": test_3m["Date"].dt.strftime("%Y-%m-%d")})
for i, col in enumerate(target_cols):
    predictions[f"baseline_CIR_{col}"] = baseline[:, i]
    predictions[f"CIRpp_{col}"] = cirpp[:, i]

predictions.to_csv("cir_predictions.csv", index=False)
predictions.head()

## Out-of-Sample Evaluation

The reconstructed yields are compared with the maturities available in `test_data.csv`. The main metric is out-of-sample R2, with RMSE also reported for readability. The extension is expected to improve the baseline and meet the required R2 threshold.

In [ ]:
def r2_score(actual: np.ndarray, predicted: np.ndarray) -> float:
    return float(1 - np.sum((actual - predicted) ** 2) / np.sum((actual - actual.mean()) ** 2))

shared_cols = [col for col in target_cols if col in test.columns]
shared_idx = [target_cols.index(col) for col in shared_cols]
actual = test[shared_cols].to_numpy(dtype=float)
base_eval = baseline[:, shared_idx]
pp_eval = cirpp[:, shared_idx]

summary_rows = []
for j, col in enumerate(shared_cols):
    summary_rows.append({
        "maturity": col,
        "baseline_r2": r2_score(actual[:, j], base_eval[:, j]),
        "cirpp_r2": r2_score(actual[:, j], pp_eval[:, j]),
        "cirpp_rmse": np.sqrt(np.mean((actual[:, j] - pp_eval[:, j]) ** 2)),
    })

summary_rows.append({
    "maturity": "ALL_TEST_MATURITIES",
    "baseline_r2": r2_score(actual.reshape(-1), base_eval.reshape(-1)),
    "cirpp_r2": r2_score(actual.reshape(-1), pp_eval.reshape(-1)),
    "cirpp_rmse": np.sqrt(np.mean((actual - pp_eval) ** 2)),
})

metrics = pd.DataFrame(summary_rows)
metrics.to_csv("cir_metrics.csv", index=False)
metrics

## Final Notes

This approach is not autoregressive. It does not use lagged target yields, previous predictions, ARIMA-style dynamics, or any future test information. Each test-date curve is reconstructed from the same-date 3M yield and the calibrated CIR/CIR++ term-structure mapping.